# DSA5206 Part II

Critical assessment and an executed patient-grouped model comparison for Cohen et al. (2020). The report and checklist.csv apply the 12 classroom questions in Lecture 9. All tables below read the actual saved experiment outputs.

The full experiment uses two targets, nine models and 50 outer patient-grouped splits. Alternative parameters are selected in five grouped inner folds, with all scalers fitted inside the pipeline. No predictions are clipped.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    raise RuntimeError('Open this notebook from the extracted package directory.')
sys.path.insert(0, str(ROOT / 'code'))
meta = pd.read_csv(ROOT / 'data/scored_metadata.csv')
features = np.load(ROOT / 'data/frozen_features.npz', allow_pickle=False)
print(f'{len(meta)} images; {meta.patientid.nunique()} patients')
print('Feature shapes:', features['logits'].shape, features['features'].shape)
print(json.loads((ROOT / 'results/protocol.json').read_text())['outer'])

94 images; 65 patients
Feature shapes: (94, 18) (94, 1024)
50 GroupShuffleSplit seeds 0 through 49; 50 percent of patient groups for test


## Verify the existing results

Verification recomputes MAE, MSE and R-squared from every stored outer-test prediction and checks patient separation. It does not refit the models.

In [2]:
from verify_results import main as verify
verify()

PASS: feature checksum; 94 images and 65 patients; no train/test patient overlap; all 900 recorded MAE, MSE and R2 values match predictions.


## Optional complete refit

Set RERUN to True to regenerate the full comparison from the packaged frozen features. Installing requirements-analysis.txt is sufficient; no image download or PyTorch is required for this step. Run code/prepare_features.py separately if you also want to recreate the features from raw images.

In [3]:
RERUN = False
if RERUN:
    from run_comparison import main as run
    run()
    from make_figures import main as plot
    plot()
else:
    print('Using saved results from the completed 900-fit experiment.')

Using saved results from the completed 900-fit experiment.


In [4]:
summary = pd.read_csv(ROOT / 'results/summary.csv')
print(summary[['target','model','MAE_mean','MAE_std','R2_mean','Pearson_mean']].round(4).to_string(index=False))

           target              model  MAE_mean  MAE_std  R2_mean  Pearson_mean
Geographic Extent      Mean baseline    2.0724   0.1665  -0.0868           NaN
Geographic Extent    Published OLS 1    1.1652   0.1040   0.5979        0.8034
Geographic Extent    Published OLS 4    1.1837   0.1206   0.5983        0.8046
Geographic Extent   Published OLS 18    1.3230   0.1962   0.4845        0.7657
Geographic Extent Published OLS 1024    1.3552   0.1268   0.4588        0.7383
Geographic Extent            Ridge 4    1.2133   0.1278   0.5861        0.8001
Geographic Extent           Ridge 18    1.2411   0.1927   0.5509        0.7862
Geographic Extent         Ridge 1024    1.4186   0.6332  -1.8938        0.7665
Geographic Extent          RBF SVR 4    1.2720   0.2017   0.5115        0.7591
          Opacity      Mean baseline    1.2886   0.1039  -0.0841           NaN
          Opacity    Published OLS 1    0.7884   0.0621   0.5852        0.7884
          Opacity    Published OLS 4    0.7663   0.0

## Compare against the strongest published reference for each target

The extent reference is Published OLS 1. The opacity reference is Published OLS 4 because its MAE was lowest in the original paper. A negative MAE difference favors the alternative. Mean differences and win fractions across overlapping splits are descriptive, not independent hypothesis tests.

In [5]:
paired = pd.read_csv(ROOT / 'results/paired_comparisons.csv')
chosen = paired[((paired.target == 'Geographic Extent') & (paired.baseline == 'Published OLS 1')) | ((paired.target == 'Opacity') & (paired.baseline == 'Published OLS 4'))]
print(chosen[['target','alternative','baseline','image_MAE_difference','split_win_fraction']].round(4).to_string(index=False))
print('Conclusion: none of the tested alternatives beat the strongest published low-dimensional OLS configuration on mean held-out MAE.')

           target alternative        baseline  image_MAE_difference  split_win_fraction
Geographic Extent     Ridge 4 Published OLS 1                0.0481                0.32
Geographic Extent    Ridge 18 Published OLS 1                0.0759                0.34
Geographic Extent  Ridge 1024 Published OLS 1                0.2534                0.28
Geographic Extent   RBF SVR 4 Published OLS 1                0.1069                0.26
          Opacity     Ridge 4 Published OLS 4                0.0037                0.60
          Opacity    Ridge 18 Published OLS 4                0.0630                0.24
          Opacity  Ridge 1024 Published OLS 4                0.1646                0.28
          Opacity   RBF SVR 4 Published OLS 4                0.1267                0.04
Conclusion: none of the tested alternatives beat the strongest published low-dimensional OLS configuration on mean held-out MAE.


## Inspect high-score errors and the retained failure

The high-score strata are extent >=6 and opacity >=4. They are exploratory radiographic groupings, not validated clinical categories. Ridge 1024 has a retained extreme prediction. The diagnostic uses a different numerical solver and confirms that a nearly constant training feature was greatly amplified by standardization.

In [6]:
print(summary[['target','model','High_score_MAE_mean','High_score_bias_mean']].round(4).to_string(index=False))
print(json.dumps(json.loads((ROOT / 'results/failure_diagnostic.json').read_text()), indent=2))

           target              model  High_score_MAE_mean  High_score_bias_mean
Geographic Extent      Mean baseline               3.7614               -3.7614
Geographic Extent    Published OLS 1               1.6118               -1.2805
Geographic Extent    Published OLS 4               1.6245               -1.4473
Geographic Extent   Published OLS 18               1.5664               -1.0510
Geographic Extent Published OLS 1024               1.5026               -0.9932
Geographic Extent            Ridge 4               1.6889               -1.5274
Geographic Extent           Ridge 18               1.6633               -1.4215
Geographic Extent         Ridge 1024               2.0786               -1.9508
Geographic Extent          RBF SVR 4               1.9039               -1.6365
          Opacity      Mean baseline               2.0398               -2.0398
          Opacity    Published OLS 1               0.8773               -0.7024
          Opacity    Published OLS 4    

![model_comparison.png](results/model_comparison.png)

![high_score_bias.png](results/high_score_bias.png)

## Interpretation and limits

Ridge improves the unregularized 18-output comparator but does not surpass the strongest low-dimensional published model. Ridge 4 has slightly lower opacity MAE than OLS 1, but OLS 4 remains lower. The tested standardized Ridge 1024 pipeline is unstable on one outer split; its failure is retained rather than hidden or clipped.

All tests are internal to the same small cohort. The 50 splits are dependent. Patient-bootstrap intervals condition on stored predictions and do not include full training variability or independent external validation. Missing age and sex values are not imputed because those variables are not used by these models.

Sources and exact source commits are documented in README.md and data/provenance.json.